# Confidence intervals on sigma_bio by cell bootstrap

This notebook reproduces the error bars on the SABER-FISH sigma_bio time series (Fig. 3G, S2F-G of the paper) from the per-cell inferred phases. It needs no GPU and no model fitting, and runs in seconds.

How the interval is built:
- `sigma_tech` of each time point comes from the twin dataset and is held **fixed**. It is not resampled.
- The cells of each time point are resampled with replacement, at the original sample size. `sigma_data` is recomputed on each resample.
- `sigma_bio = sqrt(sigma_data^2 - sigma_tech^2)`, set to 0 when `sigma_tech` exceeds `sigma_data`.
- The spread of `sigma_bio` over the resamples is the error. The slope, the late minus early contrast and the period spread `sigma_T` are recomputed on every resample.

Inputs (in `tutorials/data/smFISH/`): `smFISH_results2_v2.csv` (one row per cell, inferred phase `post_mode` in radians and `ext_time_hours`) and `desync_v2.csv` (`sigma_tech` per time point, `Technical_cSTD`, in hours).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import norm

DATA = "data/smFISH/"
N_BOOT, SEED = 2000, 1
EXCLUDE = [10.0]            # time point left out of the slope fit and the early set
LATE_FROM, EARLY_TO = 34.0, 22.0

W = 2 * np.pi / 24          # rad per hour
cells = pd.read_csv(DATA + "smFISH_results2_v2.csv")
tech = pd.read_csv(DATA + "desync_v2.csv").sort_values("sample_name")
tps = tech["sample_name"].to_numpy(float)
sig_tech = tech["Technical_cSTD"].to_numpy(float)
phases = {t: cells.loc[cells["ext_time_hours"] == t, "post_mode"].to_numpy() for t in tps}

## Helper functions

In [2]:
def circ_std_h(theta):
    """Circular standard deviation, sqrt(-2 log R), converted to hours."""
    R = np.abs(np.exp(1j * theta).mean())
    return np.sqrt(-2 * np.log(R)) / W

def sigma_bio(sigma_data):
    return np.sqrt(np.maximum(sigma_data**2 - sig_tech**2, 0))

keep = ~np.isin(tps, EXCLUDE)
late = tps >= LATE_FROM
early = (tps <= EARLY_TO) & keep

def fit_slope(y):
    """Least squares slope of sigma_bio against time, constrained to be >= 0."""
    f = lambda t, a, b: a * t + b
    return curve_fit(f, tps[keep], y[keep], p0=[0.01, 0],
                     bounds=([0, -np.inf], [np.inf, np.inf]))[0][0]

def sigma_T(slope, n=1000):
    """Period spread in hours from the slope (h per h), via T = 2 pi / omega."""
    if slope <= 0:
        return np.nan
    so = slope * W
    T = np.linspace(2 * np.pi / (W + 4 * so), 2 * np.pi / max(1e-3, W - 4 * so), n)
    pdf = norm.pdf(2 * np.pi / T, loc=W, scale=so) * 2 * np.pi / T**2
    pdf /= np.trapezoid(pdf, T)
    m = np.trapezoid(T * pdf, T)
    return np.sqrt(np.trapezoid(T**2 * pdf, T) - m**2)

def summary(bio):
    s = fit_slope(bio)
    return s, bio[late].mean() - bio[early].mean(), sigma_T(s)

## Point estimate and bootstrap

In [3]:
data_pt = np.array([circ_std_h(phases[t]) for t in tps])
bio_pt = sigma_bio(data_pt)
point = summary(bio_pt)

rng = np.random.default_rng(SEED)
data_b = np.array([[circ_std_h(p[rng.integers(0, len(p), len(p))]) for p in phases.values()]
                   for _ in range(N_BOOT)])
bio_b = sigma_bio(data_b)
boot = np.array([summary(b) for b in bio_b])

In [4]:
names = ["slope (h per h)", "late minus early (h)", "sigma_T (h)"]
out = pd.DataFrame({"point": point,
                    "median": np.nanmedian(boot, 0),
                    "2.5%": np.nanpercentile(boot, 2.5, 0),
                    "97.5%": np.nanpercentile(boot, 97.5, 0)}, index=names)
out.round(3)

,point,median,2.5%,97.5%
slope (h per h),0.056,0.051,0.026,0.071
late minus early (h),2.077,1.916,1.137,2.414
sigma_T (h),1.361,1.233,0.631,1.736


The per time point bands on `sigma_data` and `sigma_bio` (the shaded bands of Fig. 3G) are the 2.5 and 97.5 percentiles of `data_b` and `bio_b` over the resamples.

In [5]:
bands = pd.DataFrame({"time": tps, "n_cells": [len(phases[t]) for t in tps],
                      "sigma_tech": sig_tech, "sigma_data": data_pt, "sigma_bio": bio_pt,
                      "sigma_bio_lo": np.percentile(bio_b, 2.5, 0),
                      "sigma_bio_hi": np.percentile(bio_b, 97.5, 0)})
bands.round(2)

,time,n_cells,sigma_tech,sigma_data,sigma_bio,sigma_bio_lo,sigma_bio_hi
0,10.0,278,2.01,3.96,3.41,3.05,3.76
1,14.0,266,2.40,2.75,1.33,0.59,1.84
2,18.0,116,3.87,3.84,0.00,0.00,2.43
3,22.0,352,2.37,3.28,2.27,1.93,2.57
4,26.0,235,2.48,3.39,2.32,1.87,2.71
5,29.0,362,3.22,3.73,1.88,0.95,2.53
6,34.0,289,2.52,4.23,3.40,2.88,3.90
7,38.0,278,2.25,4.75,4.18,3.70,4.68
8,42.0,217,3.83,5.01,3.24,2.08,4.29
9,46.0,237,2.36,4.79,4.17,3.73,4.65
